In [0]:
CREATE OR REPLACE TABLE data_governance.gold_lineage.dim_jobs_metadata 
USING DELTA
AS
SELECT
    job_id,
    workspace_id,
    name,
    creator_user_name,
    run_as_user_name,
    job_category,
    create_time,
    CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_lineage_analysis.lineage_workflow_metadata

In [0]:
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_lineage.fact_lineage_table_flow
),

new_data AS (
    SELECT *
    FROM data_governance.silver_lineage_analysis.lineage_table_level
    WHERE _metadata.file_modification_time > (SELECT last_ts FROM last_run)
)

INSERT INTO data_governance.gold_lineage.fact_lineage_table_flow
SELECT
    account_id,
    workspace_id,
    source_table_full_name,
    target_table_full_name,
    entity_type,
    entity_run_id,
    created_by,
    event_date,
    CURRENT_TIMESTAMP() AS load_timestamp
FROM new_data;

In [0]:
-- STEP 1: Get last processed timestamp
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_lineage.fact_lineage_column_flow
),

-- STEP 2: Get new lineage records
new_data AS (
    SELECT *
    FROM data_governance.silver_lineage_analysis.lineage_column_level
    WHERE  _metadata.file_modification_time > (SELECT last_ts FROM last_run)
),

-- STEP 3: Transform
transformed AS (
    SELECT
        account_id,
        workspace_id,

        entity_type,
        entity_run_id,

        source_table_full_name,
        source_column_name,

        target_table_full_name,
        target_column_name,

        CURRENT_TIMESTAMP() AS load_timestamp

    FROM new_data
)

-- STEP 4: INSERT
INSERT INTO data_governance.gold_lineage.fact_lineage_column_flow
SELECT * FROM transformed;

In [0]:
-- working fine
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_lineage.fact_job_performance
),

new_data AS (
    SELECT *
    FROM data_governance.silver_cost_monitoring.job_run
    WHERE _metadata.file_modification_time > (SELECT last_ts FROM last_run)
),

transformed AS (
    SELECT
        j.account_id,
        j.workspace_id,
        j.job_id,
        m.name AS job_name,

        j.run_id,
        j.run_type,
        j.trigger_type,

        j.result_state,
        j.termination_type,

        j.period_start_time,
        j.period_end_time,

        j.run_duration_seconds,
        j.queue_duration_seconds,
        j.setup_duration_seconds,
        j.cleanup_duration_seconds,

        CURRENT_TIMESTAMP() AS load_timestamp

    FROM new_data j
    LEFT JOIN data_governance.gold_lineage.dim_jobs_metadata m
        ON j.job_id = m.job_id
        AND j.workspace_id = m.workspace_id
)

INSERT INTO data_governance.gold_lineage.fact_job_performance
SELECT * FROM transformed;

In [0]:
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_lineage.fact_query_info
),

new_data AS (
    SELECT *
    FROM data_governance.silver_lineage_analysis.lineage_query_history
    WHERE load_timestamp > (SELECT last_ts FROM last_run)
)

INSERT INTO data_governance.gold_lineage.fact_query_info
SELECT
    account_id,
    workspace_id,
    statement_id,
    executed_by,
    session_id,
    execution_status,
    statement_text,
    statement_type,
    total_duration_ms,
    execution_duration_ms,
    compilation_duration_ms,
    client_application,
    start_time,
    end_time,
    read_files,
    read_rows,
    produced_rows,
    read_bytes,
    pipeline_id,
    event_date,
    query_category,
    is_failed_query,
    CURRENT_TIMESTAMP() AS load_timestamp
FROM new_data;